In [1]:
# ============================================================
# TRUSTSYN STEP 4 — SHAP EXPLAINABILITY (FAST VERSION)
# ============================================================

import os
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

STACKING_DIR = "/Users/konuri/stacking/FINAL_STACKING_TABLES"

SHAP_DIR = os.path.join(
    STACKING_DIR,
    "SHAP_RESULTS"
)

os.makedirs(
    SHAP_DIR,
    exist_ok=True
)


splits = [
    "RANDOM",
    "COLD_COMBINATION",
    "COLD_CELL_LINE",
    "COLD_DRUG"
]


# ------------------------------------------------------------
# LOOP THROUGH SPLITS
# ------------------------------------------------------------

for split in splits:

    print("\n" + "="*70)
    print(split)
    print("="*70)


    file = os.path.join(
        STACKING_DIR,
        f"{split}_CatBoost_DMPNN_stacking_table.csv"
    )


    df = pd.read_csv(file)

    print("Loaded:", df.shape)


    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------

    y = df["y_true"]


    # --------------------------------------------------------
    # REMOVE IDENTIFIERS
    # --------------------------------------------------------

    drop_cols = [
        "drug_A",
        "drug_B",
        "CELLNAME",
        "tissue",
        "combo_score",
        "y_true"
    ]


    X = df.drop(
        columns=drop_cols,
        errors="ignore"
    )


    # Encode categorical columns

    for col in X.columns:

        if X[col].dtype == "object":

            X[col] = (
                X[col]
                .astype("category")
                .cat.codes
            )


    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)


    print(
        "Features:",
        X.shape
    )


# --------------------------------------------------------
# FAST META MODEL FOR EXPLAINABILITY
# --------------------------------------------------------

from sklearn.ensemble import ExtraTreesRegressor


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


meta = ExtraTreesRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
    max_depth=12
)


print("Training meta model...")


meta.fit(
    X_train,
    y_train
)


print("Meta model trained")


# --------------------------------------------------------
# SHAP
# --------------------------------------------------------

X_shap = X.sample(
    n=min(1000, len(X)),
    random_state=42
)


print("Starting SHAP...")


explainer = shap.TreeExplainer(
    meta
)


shap_values = explainer.shap_values(
    X_shap
)


print(
    "SHAP calculated:",
    shap_values.shape
)


# --------------------------------------------------------
# SAVE IMPORTANCE
# --------------------------------------------------------

shap_importance = pd.DataFrame(
    {
        "feature": X_shap.columns,
        "mean_abs_SHAP":
            np.abs(shap_values).mean(axis=0)
    }
)


shap_importance = shap_importance.sort_values(
    "mean_abs_SHAP",
    ascending=False
)


out_csv = os.path.join(
    SHAP_DIR,
    f"{split}_SHAP_feature_importance.csv"
)


shap_importance.to_csv(
    out_csv,
    index=False
)


print("Saved:", out_csv)


# --------------------------------------------------------
# PLOT
# --------------------------------------------------------

plt.figure(figsize=(10,8))


shap.summary_plot(
    shap_values,
    X_shap,
    show=False
)


plt.tight_layout()


out_png = os.path.join(
    SHAP_DIR,
    f"{split}_SHAP_summary.png"
)


plt.savefig(
    out_png,
    dpi=300,
    bbox_inches="tight"
)


plt.close()


print("Saved:", out_png)

/Users/konuri/Downloads/miniconda3/envs/FDS/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



RANDOM
Loaded: (29408, 73)
Features: (29408, 67)

COLD_COMBINATION
Loaded: (29558, 73)
Features: (29558, 67)

COLD_CELL_LINE
Loaded: (29787, 73)
Features: (29787, 67)

COLD_DRUG
Loaded: (3132, 73)
Features: (3132, 67)
Training meta model...
Meta model trained
Starting SHAP...
SHAP calculated: (1000, 67)
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/COLD_DRUG_SHAP_feature_importance.csv
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/COLD_DRUG_SHAP_summary.png


In [2]:
# ============================================================
# TRUSTSYN STEP 4 — SHAP FOR REMAINING SPLITS
# RANDOM + COLD_COMBINATION + COLD_CELL_LINE
# ============================================================

import os
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split


STACKING_DIR = "/Users/konuri/stacking/FINAL_STACKING_TABLES"

SHAP_DIR = os.path.join(
    STACKING_DIR,
    "SHAP_RESULTS"
)

os.makedirs(
    SHAP_DIR,
    exist_ok=True
)


splits = [
    "RANDOM",
    "COLD_COMBINATION",
    "COLD_CELL_LINE"
]


for split in splits:

    print("\n" + "="*70)
    print(split)
    print("="*70)


    file = os.path.join(
        STACKING_DIR,
        f"{split}_CatBoost_DMPNN_stacking_table.csv"
    )


    df = pd.read_csv(file)

    print("Loaded:", df.shape)


    y = df["y_true"]


    drop_cols = [
        "drug_A",
        "drug_B",
        "CELLNAME",
        "tissue",
        "combo_score",
        "y_true"
    ]


    X = df.drop(
        columns=drop_cols,
        errors="ignore"
    )


    # encode categorical columns

    for col in X.columns:

        if X[col].dtype == "object":

            X[col] = (
                X[col]
                .astype("category")
                .cat.codes
            )


    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)


    print(
        "Features:",
        X.shape
    )


    # -------------------------------
    # META MODEL
    # -------------------------------

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )


    print("Training meta model...")


    meta = ExtraTreesRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1,
        max_depth=12
    )


    meta.fit(
        X_train,
        y_train
    )


    print("Meta model trained")


    # -------------------------------
    # SHAP
    # -------------------------------

    X_shap = X.sample(
        n=min(1000, len(X)),
        random_state=42
    )


    print("Starting SHAP...")


    explainer = shap.TreeExplainer(
        meta
    )


    shap_values = explainer.shap_values(
        X_shap
    )


    print(
        "SHAP calculated:",
        shap_values.shape
    )


    # -------------------------------
    # SAVE CSV
    # -------------------------------

    shap_importance = pd.DataFrame(
        {
            "feature": X_shap.columns,
            "mean_abs_SHAP":
                np.abs(shap_values).mean(axis=0)
        }
    )


    shap_importance = shap_importance.sort_values(
        "mean_abs_SHAP",
        ascending=False
    )


    csv_out = os.path.join(
        SHAP_DIR,
        f"{split}_SHAP_feature_importance.csv"
    )


    shap_importance.to_csv(
        csv_out,
        index=False
    )


    print(
        "Saved:",
        csv_out
    )


    # -------------------------------
    # SAVE PLOT
    # -------------------------------

    plt.figure(
        figsize=(10,8)
    )


    shap.summary_plot(
        shap_values,
        X_shap,
        show=False
    )


    plt.tight_layout()


    png_out = os.path.join(
        SHAP_DIR,
        f"{split}_SHAP_summary.png"
    )


    plt.savefig(
        png_out,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        "Saved:",
        png_out
    )


print("\n================================")
print("REMAINING SHAP RUN COMPLETE")
print("================================")


RANDOM
Loaded: (29408, 73)
Features: (29408, 67)
Training meta model...
Meta model trained
Starting SHAP...
SHAP calculated: (1000, 67)
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/RANDOM_SHAP_feature_importance.csv
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/RANDOM_SHAP_summary.png

COLD_COMBINATION
Loaded: (29558, 73)
Features: (29558, 67)
Training meta model...
Meta model trained
Starting SHAP...
SHAP calculated: (1000, 67)
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/COLD_COMBINATION_SHAP_feature_importance.csv
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/COLD_COMBINATION_SHAP_summary.png

COLD_CELL_LINE
Loaded: (29787, 73)
Features: (29787, 67)
Training meta model...
Meta model trained
Starting SHAP...
SHAP calculated: (1000, 67)
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/COLD_CELL_LINE_SHAP_feature_importance.csv
Saved: /Users/konuri/stacking/FINAL_STACKING_TABLES/SHAP_RESULTS/